# บทที่ 3: สถาปัตยกรรมเพอร์เซ็ปตรอนหลายชั้น (Multi-Layer Perceptron: MLP)

ใน Notebook นี้ เราจะเรียนรู้โครงสร้างของเพอร์เซ็ปตรอนหลายชั้น การส่งผ่านสัญญาณไปข้างหน้า (Forward Propagation) การแก้ปัญหา XOR และการคำนวณพารามิเตอร์

**ศัพท์ที่สำคัญในบทนี้:**

- เวกเตอร์ (vector) — อาร์เรย์หนึ่งมิติ
- เมทริกซ์ (matrix) — อาร์เรย์สองมิติ
- ค่าน้ำหนัก (weight) — พารามิเตอร์ที่ปรับได้
- ค่าไบแอส (bias) — ค่าเลื่อน
- อินพุต (input) — ข้อมูลนำเข้า
- เอาต์พุต (output) — ผลลัพธ์
- ค่าสูญเสีย (loss) — วัดความคลาดเคลื่อน
- เกรเดียนต์ (gradient) — ทิศทางการปรับ
- ฟังก์ชันกระตุ้น (activation function) — ฟังก์ชันไม่เป็นเชิงเส้น
- โครงข่ายประสาทเทียม (neural network) — โมเดลแมชชีนเลิร์นนิง

## 1. นำเข้าไลบรารีที่จำเป็น

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

# ติดตั้งฟอนต์ภาษาไทยสำหรับ Google Colab
import subprocess, glob
try:
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'fonts-tlwg-garuda'],
                   capture_output=True)
except FileNotFoundError:
    pass  # เครื่องที่ไม่มี apt-get (macOS/Windows) ใช้ฟอนต์ไทยที่ติดตั้งไว้ในเครื่องแทน

# ลงทะเบียนฟอนต์โดยตรง
from matplotlib.font_manager import fontManager
for font_file in glob.glob('/usr/share/fonts/truetype/tlwg/*.ttf'):
    fontManager.addfont(font_file)

# ตั้งค่า Seaborn theme และฟอนต์ภาษาไทย
sns.set_theme(style='whitegrid', font='Garuda')
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
%config InlineBackend.figure_format = 'retina'

from matplotlib.colors import ListedColormap

## 2. ข้อจำกัดของ Perceptron เดี่ยวและปัญหา XOR

In [ ]:
# แสดงปัญหา XOR
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([0, 1, 1, 0])

plt.figure(figsize=(8, 6))
colors = ['red', 'blue']
for i, (xi, yi) in enumerate(zip(X, y_xor)):
    plt.scatter(xi[0], xi[1], c=colors[yi], s=200, edgecolors='black', linewidths=2)
    plt.annotate(f'({xi[0]}, {xi[1]}) → {yi}', 
                 xy=(xi[0], xi[1]), xytext=(10, 10), 
                 textcoords='offset points', fontsize=12)

plt.xlabel('x₁')
plt.ylabel('x₂')
plt.title('ปัญหา XOR: ไม่สามารถแยกด้วยเส้นตรง')
plt.xlim(-0.5, 1.5)
plt.ylim(-0.5, 1.5)
plt.show()

print("สังเกต: ไม่มีเส้นตรงที่สามารถแยก class 0 และ class 1 ได้")

## 3. โครงสร้างของ MLP

เพอร์เซ็ปตรอนหลายชั้นประกอบด้วย:
- ชั้นอินพุต (Input Layer)
- ชั้นซ่อน (Hidden Layer) หนึ่งชั้นขึ้นไป
- ชั้นเอาต์พุต (Output Layer)

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1 - s)

class MLP:
    """
    เพอร์เซ็ปตรอนหลายชั้น (Multi-Layer Perceptron)
    ตัวอย่างนี้ใช้ค่าสูญเสียกำลังสอง (MSE) เพียงเพื่อสาธิตการฝึกด้วยเกรเดียนต์ดีเซนต์
    (ฟังก์ชันสูญเสียและการแพร่กระจายย้อนกลับอย่างเป็นทางการอธิบายในบทถัดไป)
    """
    def __init__(self, layer_sizes, learning_rate=0.5):
        """
        พารามิเตอร์:
        - layer_sizes: list ของจำนวนเซลล์ประสาทในแต่ละชั้น
          เช่น [2, 4, 1] = อินพุต 2, ชั้นซ่อน 4, เอาต์พุต 1
        """
        self.layer_sizes = layer_sizes
        self.lr = learning_rate
        self.n_layers = len(layer_sizes)

        # กำหนดค่าน้ำหนักและค่าไบแอสเริ่มต้น
        self.weights = []
        self.biases = []

        for i in range(self.n_layers - 1):
            w = np.random.randn(layer_sizes[i+1], layer_sizes[i]) * np.sqrt(2/layer_sizes[i])
            b = np.zeros((layer_sizes[i+1], 1))
            self.weights.append(w)
            self.biases.append(b)

    def forward(self, x):
        """การส่งผ่านสัญญาณไปข้างหน้า"""
        self.activations = [x.reshape(-1, 1)]
        self.z_values = []

        current = self.activations[0]
        for i in range(self.n_layers - 1):
            z = np.dot(self.weights[i], current) + self.biases[i]
            self.z_values.append(z)
            current = sigmoid(z)
            self.activations.append(current)

        return current

    def backward(self, y):
        """การแพร่กระจายย้อนกลับ"""
        y = y.reshape(-1, 1)
        m = 1

        # เกรเดียนต์ของชั้นเอาต์พุต: dL/dz = (a - y) * f'(z) ตาม chain rule ของค่าสูญเสียกำลังสอง
        delta = (self.activations[-1] - y) * sigmoid_derivative(self.z_values[-1])

        self.dW = []
        self.db = []

        for i in range(self.n_layers - 2, -1, -1):
            dW = np.dot(delta, self.activations[i].T)
            db = delta

            self.dW.insert(0, dW)
            self.db.insert(0, db)

            if i > 0:
                delta = np.dot(self.weights[i].T, delta) * sigmoid_derivative(self.z_values[i-1])

    def update_weights(self):
        """ปรับค่าน้ำหนักด้วยการไล่ลงตามเกรเดียนต์"""
        for i in range(self.n_layers - 1):
            self.weights[i] -= self.lr * self.dW[i]
            self.biases[i] -= self.lr * self.db[i]

    def train(self, X, y, epochs=10000, verbose=True):
        """ฝึกโครงข่าย"""
        for epoch in range(epochs):
            for xi, yi in zip(X, y):
                self.forward(xi)
                self.backward(yi)
                self.update_weights()

            if verbose and epoch % 2000 == 0:
                loss = self.compute_loss(X, y)
                print(f"รอบที่ {epoch}: ค่าสูญเสีย = {loss:.6f}")

    def predict(self, x):
        """ทำนาย"""
        return self.forward(x)[0, 0]

    def compute_loss(self, X, y):
        """คำนวณค่าสูญเสียกำลังสองเฉลี่ย (MSE)"""
        total_loss = 0
        for xi, yi in zip(X, y):
            pred = self.predict(xi)
            total_loss += (yi - pred)**2
        return total_loss / len(X)

## 4. การแก้ปัญหา XOR ด้วย MLP

In [ ]:
# สร้าง MLP สำหรับ XOR
mlp = MLP(layer_sizes=[2, 4, 1], learning_rate=0.5)

# ฝึกโครงข่าย
print("=== ฝึก MLP สำหรับ XOR ===")
mlp.train(X, y_xor, epochs=10000)

# ทดสอบ
print("\n=== ผลลัพธ์ ===")
for xi, yi in zip(X, y_xor):
    pred = mlp.predict(xi)
    print(f"อินพุต: {xi}, คำตอบจริง: {yi}, ค่าทำนาย: {pred:.4f} → {1 if pred > 0.5 else 0}")

## 5. แสดงขอบเขตการตัดสินใจ (Decision Boundary)

In [ ]:
def plot_decision_boundary(mlp, X, y, title):
    """แสดงขอบเขตการตัดสินใจ"""
    x_min, x_max = -0.5, 1.5
    y_min, y_max = -0.5, 1.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100),
                         np.linspace(y_min, y_max, 100))

    Z = np.array([mlp.predict(np.array([x, y]))
                  for x, y in zip(xx.ravel(), yy.ravel())])
    Z = Z.reshape(xx.shape)

    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap=ListedColormap(['red', 'blue']))
    plt.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)

    for i, (xi, yi) in enumerate(zip(X, y)):
        plt.scatter(xi[0], xi[1], c=['red', 'blue'][yi], s=200,
                   edgecolors='black', linewidths=2)

    plt.xlabel('x₁')
    plt.ylabel('x₂')
    plt.title(title)
    plt.show()

plot_decision_boundary(mlp, X, y_xor, 'ขอบเขตการตัดสินใจของ MLP สำหรับ XOR')

## 6. การคำนวณจำนวนพารามิเตอร์

In [ ]:
def count_parameters(layer_sizes):
    """
    คำนวณจำนวนพารามิเตอร์ใน MLP

    พารามิเตอร์ = ค่าน้ำหนัก + ค่าไบแอส
    - ค่าน้ำหนัก: n_in × n_out
    - ค่าไบแอส: n_out
    """
    total = 0
    print(f"ขนาดแต่ละชั้น: {layer_sizes}")
    print("\n" + "="*50)

    for i in range(len(layer_sizes) - 1):
        n_in = layer_sizes[i]
        n_out = layer_sizes[i+1]

        weights = n_in * n_out
        biases = n_out
        layer_params = weights + biases
        total += layer_params

        print(f"ชั้น {i} → ชั้น {i+1}:")
        print(f"  ค่าน้ำหนัก: {n_in} × {n_out} = {weights}")
        print(f"  ค่าไบแอส: {biases}")
        print(f"  รวมย่อย: {layer_params}")
        print("-" * 30)

    print(f"\n>>> พารามิเตอร์ทั้งหมด: {total}")
    return total

# ตัวอย่าง
print("=== MLP สำหรับ XOR ===")
_ = count_parameters([2, 4, 1])

print("\n\n=== MLP ขนาดใหญ่ ===")
_ = count_parameters([784, 128, 64, 10])

## 7. ตัวอย่างการส่งผ่านสัญญาณไปข้างหน้า: ReLU ในชั้นซ่อน + ซิกมอยด์ในชั้นเอาต์พุต

บทที่ 3 แสดงตัวอย่าง MLP ขนาด $3 \rightarrow 4 \rightarrow 2$ ที่ชั้นซ่อนใช้ ReLU และชั้นเอาต์พุตใช้ซิกมอยด์ ต่างจากคลาส `MLP` ด้านบนที่ใช้ซิกมอยด์ทุกชั้นเพื่อความง่ายในการฝึกด้วยเกรเดียนต์ดีเซนต์ ตัวอย่างต่อไปนี้กำหนดค่าน้ำหนักและค่าไบแอสตามที่บทระบุตรง ๆ แล้วคำนวณผลลัพธ์เพื่อตรวจสอบว่าตัวเลขตรงกับที่บทคำนวณไว้

In [ ]:
def relu(x):
    """ฟังก์ชันกระตุ้น ReLU: ให้ค่าเป็นศูนย์เมื่ออินพุตเป็นลบ"""
    return np.maximum(0, x)

# เวกเตอร์อินพุตและพารามิเตอร์ตามตัวอย่างของบทที่ 3 (MLP ขนาด 3 -> 4 -> 2)
x_ex = np.array([[0.5], [0.3], [0.8]])

W1_ex = np.array([
    [0.2, -0.1, 0.4],
    [0.5, 0.3, -0.2],
    [-0.3, 0.6, 0.1],
    [0.4, -0.5, 0.2]
])
b1_ex = np.array([[0.1], [-0.3], [0.2], [0]])

W2_ex = np.array([
    [0.3, -0.4, 0.2, 0.5],
    [-0.2, 0.6, -0.3, 0.4]
])
b2_ex = np.array([[0.1], [-0.05]])

# ขั้นตอนที่ 1: คำนวณค่าก่อนกระตุ้นและค่ากระตุ้นของชั้นซ่อนด้วย ReLU
z1_ex = np.dot(W1_ex, x_ex) + b1_ex
a1_ex = relu(z1_ex)

# ขั้นตอนที่ 2: ส่ง a(1) ไปยังชั้นเอาต์พุตแล้วใช้ซิกมอยด์
z2_ex = np.dot(W2_ex, a1_ex) + b2_ex
yhat_ex = sigmoid(z2_ex)

print(f"z(1) =\n{z1_ex.ravel()}")
print(f"\na(1) = ReLU(z(1)) =\n{a1_ex.ravel()}")
print(f"\nz(2) =\n{z2_ex.ravel()}")
print(f"\nŷ = sigmoid(z(2)) =\n{yhat_ex.ravel()}")

print("\nค่าที่บทที่ 3 คำนวณไว้: z(1)=[0.49, -0.12, 0.31, 0.21], "
      "a(1)=[0.49, 0, 0.31, 0.21], z(2)=[0.414, -0.157], ŷ≈[0.6020, 0.4608]")

## 8. การประมวลผลแบบชุด (Batch Processing)

บทที่ 3 นิยามการส่งผ่านสัญญาณแบบชุดด้วย $\mathbf{Z}^{(1)}=\mathbf{W}^{(1)}\mathbf{X}+\mathbf{b}^{(1)}\mathbf{1}^T$ โดยจัดตัวอย่างแต่ละตัวเป็นหนึ่งคอลัมน์ของเมทริกซ์อินพุต $\mathbf{X}$ ตัวอย่างต่อไปนี้ส่งตัวอย่างสามตัวผ่านโครงข่ายพร้อมกันโดยใช้พารามิเตอร์ชุดเดียวกับหัวข้อก่อนหน้า แล้วตรวจสอบว่าคอลัมน์แรกของผลลัพธ์แบบชุดตรงกับผลลัพธ์ที่คำนวณทีละตัวอย่างในหัวข้อก่อนหน้าหรือไม่

In [ ]:
# เมทริกซ์อินพุต X: แต่ละคอลัมน์คือหนึ่งตัวอย่าง (3 ตัวอย่าง, 3 คุณลักษณะต่อตัวอย่าง)
# คอลัมน์แรกคือ x_ex จากตัวอย่างในหัวข้อ 7 เพื่อใช้ตรวจสอบผลลัพธ์
X_batch = np.array([
    [0.5, 0.1, 0.7],
    [0.3, 0.9, 0.2],
    [0.8, 0.4, 0.6]
])
B = X_batch.shape[1]
ones_B = np.ones((1, B))

# Z(1) = W(1) X + b(1) 1^T
Z1_batch = np.dot(W1_ex, X_batch) + np.dot(b1_ex, ones_B)
A1_batch = relu(Z1_batch)

# Z(2) = W(2) A(1) + b(2) 1^T
Z2_batch = np.dot(W2_ex, A1_batch) + np.dot(b2_ex, ones_B)
Yhat_batch = sigmoid(Z2_batch)

print(f"X ขนาด {X_batch.shape} (แถว = คุณลักษณะ, คอลัมน์ = ตัวอย่าง) =\n{X_batch}")
print(f"\nZ(1) ขนาด {Z1_batch.shape} =\n{Z1_batch}")
print(f"\nA(1) = ReLU(Z(1)) ขนาด {A1_batch.shape} =\n{A1_batch}")
print(f"\nŶ = sigmoid(Z(2)) ขนาด {Yhat_batch.shape} =\n{Yhat_batch}")

print(f"\nคอลัมน์แรกของ Ŷ: {Yhat_batch[:, 0]}")
print(f"ตรงกับ ŷ ที่คำนวณทีละตัวอย่างในหัวข้อ 7: "
      f"{np.allclose(Yhat_batch[:, 0], yhat_ex.ravel())}")

## 9. ตรวจสอบคำตอบ XOR เชิงวิเคราะห์ของบทที่ 3 (ฟังก์ชันขั้นบันได)

หัวข้อ 4 ฝึก MLP ด้วยเกรเดียนต์ดีเซนต์และซิกมอยด์ ซึ่งเป็นวิธีเชิงตัวเลขที่ต่างจากคำตอบเชิงวิเคราะห์ที่บทที่ 3 พิสูจน์ไว้ในหัวข้อกรณีศึกษา XOR บทกำหนดโครงสร้าง $2 \rightarrow 2 \rightarrow 1$ ให้ใช้ฟังก์ชันขั้นบันไดเป็นฟังก์ชันกระตุ้น และกำหนดค่าน้ำหนักกับค่าไบแอสตายตัวไว้แล้ว ตัวอย่างต่อไปนี้ใช้ค่าเหล่านั้นตรง ๆ แล้วตรวจสอบว่าได้คำตอบ XOR ถูกต้องครบทั้งสี่กรณี

In [ ]:
def step(z):
    """ฟังก์ชันขั้นบันได: ให้ค่าเป็น 1 เมื่ออินพุตไม่น้อยกว่าศูนย์ และให้ค่าเป็น 0 ในกรณีอื่น"""
    return (z >= 0).astype(int)

# ค่าน้ำหนักและค่าไบแอสตามกรณีศึกษา XOR ของบทที่ 3
# ชั้นซ่อน: เซลล์ประสาทตัวที่ 1 คำนวณ OR, เซลล์ประสาทตัวที่ 2 คำนวณ AND
W1_xor = np.array([[1, 1],
                    [1, 1]])
b1_xor = np.array([[-0.5], [-1.5]])

# ชั้นเอาต์พุต: รวมผลเป็น h1 AND (NOT h2) ซึ่งเทียบเท่า XOR
W2_xor = np.array([[1, -2]])
b2_xor = np.array([[-0.5]])

X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor_truth = [0, 1, 1, 0]

print(f"{'x1':>3} {'x2':>3} {'h1':>4} {'h2':>4} {'ŷ':>4} {'คำตอบจริง':>10}")
all_correct = True
for (x1, x2), y_true in zip(X_xor, y_xor_truth):
    x_vec = np.array([[x1], [x2]])
    z1 = np.dot(W1_xor, x_vec) + b1_xor
    h = step(z1)
    z2 = np.dot(W2_xor, h) + b2_xor
    y_pred = int(step(z2)[0, 0])
    correct = "ถูกต้อง" if y_pred == y_true else "ผิด"
    all_correct = all_correct and (y_pred == y_true)
    print(f"{x1:>3} {x2:>3} {h[0,0]:>4} {h[1,0]:>4} {y_pred:>4} {y_true:>10}  {correct}")

print(f"\nMLP เชิงวิเคราะห์ตามบทที่ 3 ให้คำตอบ XOR ถูกต้องครบทั้งสี่กรณี: {all_correct}")

## 10. แบบฝึกหัดการคำนวณ

### แบบฝึกหัดที่ 1: การส่งผ่านสัญญาณไปข้างหน้า (Forward Propagation)
ให้ MLP มีโครงสร้าง [2, 3, 1] โดย:
- W1 = [[0.5, 0.2], [0.3, 0.4], [0.1, 0.6]]
- b1 = [[0.1], [0.2], [0.3]]
- W2 = [[0.4, 0.3, 0.2]]
- b2 = [[0.1]]

จงคำนวณผลลัพธ์สำหรับ input x = [1, 0]

In [ ]:
# เขียนโค้ดที่นี่
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

x = np.array([[1], [0]])
W1 = np.array([[0.5, 0.2], [0.3, 0.4], [0.1, 0.6]])
b1 = np.array([[0.1], [0.2], [0.3]])
W2 = np.array([[0.4, 0.3, 0.2]])
b2 = np.array([[0.1]])

# การส่งผ่านสัญญาณไปข้างหน้า
z1 = np.dot(W1, x) + b1
a1 = sigmoid(z1)
z2 = np.dot(W2, a1) + b2
a2 = sigmoid(z2)

print(f"อินพุต: x = [1, 0]")
print(f"\nz1 = W1·x + b1 =\n{z1}")
print(f"\na1 = sigmoid(z1) =\n{a1}")
print(f"\nz2 = W2·a1 + b2 =\n{z2}")
print(f"\nเอาต์พุต: a2 = sigmoid(z2) = {a2[0,0]:.6f}")

### แบบฝึกหัดที่ 2: คำนวณจำนวนพารามิเตอร์
จงคำนวณจำนวนพารามิเตอร์ของ MLP ที่มีโครงสร้าง:
- ชั้นอินพุต: 10 เซลล์ประสาท
- ชั้นซ่อนที่ 1: 20 เซลล์ประสาท
- ชั้นซ่อนที่ 2: 15 เซลล์ประสาท
- ชั้นเอาต์พุต: 3 เซลล์ประสาท

In [ ]:
# เขียนโค้ดที่นี่
layer_sizes = [10, 20, 15, 3]
total = count_parameters(layer_sizes)

### แบบฝึกหัดที่ 3: ฝึก MLP สำหรับแอนเกต (AND)

In [ ]:
# เขียนโค้ดที่นี่
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_and = np.array([0, 0, 0, 1])

mlp_and = MLP(layer_sizes=[2, 2, 1], learning_rate=0.5)
mlp_and.train(X, y_and, epochs=5000)

print("\n=== ผลลัพธ์แอนเกต ===")
for xi, yi in zip(X, y_and):
    pred = mlp_and.predict(xi)
    print(f"อินพุต: {xi}, คำตอบจริง: {yi}, ค่าทำนาย: {pred:.4f}")

### แบบฝึกหัดที่ 4: ฝึก MLP สำหรับออเกต (OR)

In [ ]:
# เขียนโค้ดที่นี่
y_or = np.array([0, 1, 1, 1])

mlp_or = MLP(layer_sizes=[2, 2, 1], learning_rate=0.5)
mlp_or.train(X, y_or, epochs=5000)

print("\n=== ผลลัพธ์ออเกต ===")
for xi, yi in zip(X, y_or):
    pred = mlp_or.predict(xi)
    print(f"อินพุต: {xi}, คำตอบจริง: {yi}, ค่าทำนาย: {pred:.4f}")

## บทสรุป

Notebook นี้แสดงให้เห็นว่า:
1. **เพอร์เซ็ปตรอนเดี่ยว** ไม่สามารถแก้ปัญหา XOR ได้
2. **MLP** สามารถแก้ปัญหา XOR ได้ เพราะมีชั้นซ่อน ทั้งจากการฝึกด้วยเกรเดียนต์ดีเซนต์และจากคำตอบเชิงวิเคราะห์ที่บทที่ 3 กำหนดค่าน้ำหนัก/ไบแอสไว้ตรง ๆ ด้วยฟังก์ชันขั้นบันได
3. **การคำนวณพารามิเตอร์** สำคัญสำหรับการวางแผนโครงสร้างโมเดล
4. **การส่งผ่านสัญญาณไปข้างหน้า** ทีละตัวอย่างด้วย ReLU ในชั้นซ่อนและซิกมอยด์ในชั้นเอาต์พุต ให้ผลลัพธ์ตัวเลขตรงกับตัวอย่างในบทที่ 3
5. **การประมวลผลแบบชุด (batch processing)** ใช้เมทริกซ์อินพุต $\mathbf{X}$ ที่แต่ละคอลัมน์เป็นหนึ่งตัวอย่าง ทำให้คำนวณหลายตัวอย่างพร้อมกันได้ และให้ผลลัพธ์ตรงกับการคำนวณทีละตัวอย่าง

ผู้อ่านสามารถทดลองเปลี่ยนจำนวนเซลล์ประสาทในชั้นซ่อนและอัตราการเรียนรู้